# Notebook 2: Arquitetura e Treino da Rede Neuronal (DenseNet121)
### Projeto Final de Licenciatura - Sistemas de Apoio à Decisão Clínica

**Autor:** Gustavo P. Vília  
**Âmbito:** Transparência em Sistemas de Apoio à Decisão Clínica: Impacto da Inteligência Artificial Explicável na Triagem Radiológica

---

## 🎯 Objetivo deste Notebook
Este documento foca-se no núcleo de *Deep Learning* da investigação. O objetivo é importar a base de dados previamente balanceada (no Notebook 1), instanciar a arquitetura convolucional **DenseNet121** e treiná-la para um problema de classificação binária (Saudável *vs.* Nódulo/Massa).

## 🧠 Metodologia de Transfer Learning
De forma a maximizar a eficiência computacional e a capacidade preditiva do modelo, aplica-se a técnica de *Transfer Learning*. A rede é inicializada com os pesos pré-treinados do *dataset* genérico *ImageNet*. A camada de classificação original (concebida para 1000 classes) é removida ("*topless*") e substituída por uma nova camada densa (*Fully Connected*) com uma função de ativação *Sigmoid*, otimizada para o diagnóstico radiológico.

## 📋 Etapas Principais:
1. **Reconstrução do Ambiente:** Recuperação do *dataset* de imagens via API (KaggleHub) e importação dos ficheiros CSV de metadados (`df_treino`, `df_val`, `df_teste`).
2. **Construção do Modelo:** Instanciação da DenseNet121 e acoplamento das novas camadas de saída.
3. **Treino do Algoritmo:** Execução do processo de aprendizagem otimizado por GPU, com monitorização em tempo real da função de perda e implementação de *Early Stopping* para prevenir *overfitting*.
4. **Avaliação e Exportação:** Validação do desempenho no conjunto de teste e gravação do modelo final (`.h5`) para posterior auditoria visual (Grad-CAM no Notebook 3).

## 1. Reconstrução do Ambiente e Importação de Dados

Como o ambiente do Google Colab é volátil entre sessões, o primeiro passo deste notebook consiste na re-instanciação dos dados. Recorremos novamente à biblioteca kagglehub para transferir as imagens radiográficas nativas (224x224). Simultaneamente, efetuamos o upload do ficheiro compactado gerado na fase anterior (datasets_limpos_projeto.zip), que contém os metadados já balanceados e divididos em Treino, Validação e Teste. Uma etapa técnica crucial aqui é a re-atribuição do caminho absoluto das imagens na coluna Caminho_Imagem, garantindo que os caminhos temporários da nova máquina virtual correspondam exatamente à localização física das imagens acabadas de descarregar, prevenindo erros de leitura I/O.

In [1]:
# Instalar kagglehub se necessário
!pip install -q kagglehub

import kagglehub
import pandas as pd
import os
import zipfile
from google.colab import files

# 1. Re-importar as imagens via KaggleHub (será rápido)
print("A transferir as imagens (224x224) via KaggleHub...")
caminho_dataset = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
pasta_imagens = os.path.join(caminho_dataset, 'images-224')
print(f"✅ Imagens prontas no caminho: {pasta_imagens}\n")

# 2. Fazer o upload do ficheiro ZIP com os nossos CSVs limpos
print("Por favor, faz o upload do ficheiro 'datasets_limpos_projeto.zip' que descarregaste no Notebook 1:")
uploaded = files.upload()

# 3. Extrair os CSVs do ZIP
nome_zip = list(uploaded.keys())[0] # Pega no nome do ficheiro que acabaste de carregar
with zipfile.ZipFile(nome_zip, 'r') as zip_ref:
    zip_ref.extractall()
print("\n✅ Ficheiros CSV extraídos com sucesso!")

# 4. Carregar os dados para a memória (Pandas)
df_treino = pd.read_csv('df_treino_limpo.csv')
df_val = pd.read_csv('df_val_limpo.csv')
df_teste = pd.read_csv('df_teste_limpo.csv')

# 5. ATUALIZAÇÃO DE SEGURANÇA: Reconstruir o caminho absoluto das imagens
# Isto garante que, mesmo que o Colab mude o nome das pastas temporárias, as imagens são sempre encontradas
df_treino['Caminho_Imagem'] = df_treino['Image Index'].apply(lambda x: os.path.join(pasta_imagens, x))
df_val['Caminho_Imagem'] = df_val['Image Index'].apply(lambda x: os.path.join(pasta_imagens, x))
df_teste['Caminho_Imagem'] = df_teste['Image Index'].apply(lambda x: os.path.join(pasta_imagens, x))

# Converter os rótulos para string (exigência do Keras)
df_treino['Rotulo'] = df_treino['Rotulo'].astype(str)
df_val['Rotulo'] = df_val['Rotulo'].astype(str)
df_teste['Rotulo'] = df_teste['Rotulo'].astype(str)

print("\n📊 Resumo dos dados carregados e prontos para treino:")
print(f"Imagens de Treino: {len(df_treino)}")
print(f"Imagens de Validação: {len(df_val)}")
print(f"Imagens de Teste: {len(df_teste)}")

A transferir as imagens (224x224) via KaggleHub...
Using Colab cache for faster access to the 'nih-chest-x-ray-14-224x224-resized' dataset.
✅ Imagens prontas no caminho: /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224

Por favor, faz o upload do ficheiro 'datasets_limpos_projeto.zip' que descarregaste no Notebook 1:


Saving datasets_limpos_projeto.zip to datasets_limpos_projeto.zip

✅ Ficheiros CSV extraídos com sucesso!

📊 Resumo dos dados carregados e prontos para treino:
Imagens de Treino: 15980
Imagens de Validação: 10414
Imagens de Teste: 11098


## 2. Preparação dos Geradores de Imagem (Data Generators)
De modo a alimentar a rede neuronal de forma otimizada e sem exceder a capacidade da memória RAM, instanciamos a classe ImageDataGenerator. Este mecanismo carrega as imagens do disco em pequenos lotes (batches de 32 imagens) em tempo real. Replicando a metodologia previamente validada, aplica-se Data Augmentation exclusivamente ao conjunto de Treino para potenciar a generalização do modelo, enquanto os conjuntos de Validação e Teste são mantidos no seu estado original, sofrendo apenas a necessária normalização matemática dos píxeis.

In [2]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Deteção Automática do Caminho Real das Imagens
pasta_base = os.path.join(caminho_dataset, 'images-224')

# O Kaggle costuma criar subpastas duplicadas. Vamos verificar se as imagens estão lá dentro:
if 'images-224' in os.listdir(pasta_base):
    pasta_imagens_real = os.path.join(pasta_base, 'images-224')
else:
    pasta_imagens_real = pasta_base

print(f"🔍 Caminho real das imagens detetado: {pasta_imagens_real}")
print(f"Exemplo de ficheiros lá dentro: {os.listdir(pasta_imagens_real)[:3]}\n")

# 2. Corrigir os caminhos nos DataFrames
df_treino['Caminho_Imagem'] = df_treino['Image Index'].apply(lambda x: os.path.join(pasta_imagens_real, x))
df_val['Caminho_Imagem'] = df_val['Image Index'].apply(lambda x: os.path.join(pasta_imagens_real, x))
df_teste['Caminho_Imagem'] = df_teste['Image Index'].apply(lambda x: os.path.join(pasta_imagens_real, x))

# 3. Configurar os Geradores (As "fábricas")
TAMANHO_IMAGEM = (224, 224)
TAMANHO_LOTE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# 4. Criar os fluxos de dados a partir dos caminhos corrigidos
print("A preparar o fluxo de Treino...")
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_treino,
    x_col='Caminho_Imagem',
    y_col='Rotulo',
    target_size=TAMANHO_IMAGEM,
    batch_size=TAMANHO_LOTE,
    class_mode='binary'
)

print("\nA preparar o fluxo de Validação...")
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col='Caminho_Imagem',
    y_col='Rotulo',
    target_size=TAMANHO_IMAGEM,
    batch_size=TAMANHO_LOTE,
    class_mode='binary',
    shuffle=False
)

print("\nA preparar o fluxo de Teste...")
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_teste,
    x_col='Caminho_Imagem',
    y_col='Rotulo',
    target_size=TAMANHO_IMAGEM,
    batch_size=TAMANHO_LOTE,
    class_mode='binary',
    shuffle=False
)

🔍 Caminho real das imagens detetado: /kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224
Exemplo de ficheiros lá dentro: ['00026563_000.png', '00025583_005.png', '00006199_010.png']

A preparar o fluxo de Treino...
Found 15980 validated image filenames belonging to 2 classes.

A preparar o fluxo de Validação...
Found 10414 validated image filenames belonging to 2 classes.

A preparar o fluxo de Teste...
Found 11098 validated image filenames belonging to 2 classes.


## 3. Construção da Arquitetura (DenseNet121 e Transfer Learning)

A essência do sistema preditivo assenta na arquitetura DenseNet121. Recorrendo ao paradigma de Transfer Learning, o modelo é inicializado com a base de conhecimento (pesos sinápticos) do dataset ImageNet (weights='imagenet'). A camada classificadora superior (top layer) é descartada (include_top=False) e substituída por uma camada de Global Average Pooling seguida de uma camada Densa com um único nó e ativação sigmoid, adequada à classificação binária probabilística pretendida. Para a otimização dos gradientes, selecionou-se o algoritmo Adam com uma taxa de aprendizagem (learning rate) conservadora de 1e-4, mitigando o risco de "esquecimento catastrófico" (catastrophic forgetting) dos pesos pré-treinados. A compilação integra as métricas clínicas previamente fundamentadas: Exatidão, Precisão e Sensibilidade (Recall).

In [3]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall

# 1. Importar a base da DenseNet121 (sem a "cabeça" original)
print("A instanciar a arquitetura DenseNet121 (pesos ImageNet)...")
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Construir o "novo cérebro" (Top Layers) focado no problema radiológico
x = base_model.output
# O Global Average Pooling espalma a matriz 3D resultante da rede num vetor 1D
x = GlobalAveragePooling2D()(x)

# Camada final com 1 neurónio e ativação matemática Sigmoid (devolve um valor entre 0 e 1)
predicoes = Dense(1, activation='sigmoid')(x)

# 3. Unir a base e a nova cabeça num modelo final Keras
modelo = Model(inputs=base_model.input, outputs=predicoes)

# 4. Compilar o modelo matemático
otimizador = Adam(learning_rate=0.0001) # Learning rate baixa para proteger os pesos pré-treinados

modelo.compile(
    optimizer=otimizador,
    loss='binary_crossentropy', # A função de perda standard para problemas binários
    metrics=[
        BinaryAccuracy(name='exatidao'),
        Precision(name='precisao'),
        Recall(name='sensibilidade') # A métrica vital para não mandarmos doentes para casa!
    ]
)

print("\n✅ Arquitetura finalizada e compilada com sucesso!")
print(f"Total de parâmetros do modelo: {modelo.count_params():,}")

A instanciar a arquitetura DenseNet121 (pesos ImageNet)...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

✅ Arquitetura finalizada e compilada com sucesso!
Total de parâmetros do modelo: 7,038,529


## 4. Treino e Otimização do Modelo

A fase de treino é o núcleo computacional do projeto. O modelo é submetido ao fluxo de imagens preparado pelos geradores. Para garantir a máxima eficiência e prevenir a degradação da performance (overfitting), implementam-se três Callbacks de segurança. O ModelCheckpoint assegura a gravação persistente dos pesos da época que apresentar a menor função de perda na validação (val_loss). O EarlyStopping interrompe precocemente o treino caso o modelo estabilize e deixe de apresentar melhorias após 5 épocas, restaurando automaticamente a melhor versão. Por fim, o ReduceLROnPlateau atua como um mecanismo de ajuste fino dinâmico, reduzindo a taxa de aprendizagem (learning rate) se os gradientes ficarem estagnados em mínimos locais.

In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Configurar os mecanismos de segurança (Callbacks)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,             # Espera 5 épocas sem melhorias antes de parar
    restore_best_weights=True, # No fim, devolve a melhor versão da rede
    verbose=1
)

checkpoint = ModelCheckpoint(
    'melhor_modelo_densenet.h5', # O ficheiro que vamos usar no Notebook 3 para o Grad-CAM!
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,             # Reduz a learning rate para 20% do valor anterior
    patience=3,             # Se ficar encravado 3 épocas
    min_lr=1e-6,
    verbose=1
)

# 2. Definir o número máximo de épocas
# 20 épocas é um bom teto. O EarlyStopping provavelmente vai pará-lo antes de chegar ao fim.
EPOCAS = 20

# 3. O GRANDE MOMENTO: Iniciar o Treino
print("🚀 A iniciar o treino da DenseNet121! Agarra-te bem (isto vai demorar um bocado)...")

historico = modelo.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCAS,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

print("\n✅ TREINO CONCLUÍDO! O teu cérebro artificial está pronto e guardado.")

🚀 A iniciar o treino da DenseNet121! Agarra-te bem (isto vai demorar um bocado)...
Epoch 1/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 938ms/step - exatidao: 0.6323 - loss: 0.6769 - precisao: 0.6216 - sensibilidade: 0.7189
Epoch 1: val_loss improved from None to 0.66101, saving model to melhor_modelo_densenet.h5



Epoch 1: finished saving model to melhor_modelo_densenet.h5
500/500 ━━━━━━━━━━━━━━━━━━━━ 776s 1s/step - exatidao: 0.6629 - loss: 0.6272 - precisao: 0.6528 - sensibilidade: 0.6959 - val_exatidao: 0.6774 - val_loss: 0.6610 - val_precisao: 0.9255 - val_sensibilidade: 0.6761 - learning_rate: 1.0000e-04
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 551ms/step - exatidao: 0.7152 - loss: 0.5655 - precisao: 0.7077 - sensibilidade: 0.7399
Epoch 2: val_loss did not improve from 0.66101
500/500 ━━━━━━━━━━━━━━━━━━━━ 309s 617ms/step - exatidao: 0.7203 - loss: 0.5610 - precisao: 0.7103 - sensibilidade: 0.7439 - val_exatidao: 0.6153 - val_loss: 0.6858 - val_precisao: 0.9380 - val_sensibilidade: 0.5878 - learning_rate: 1.0000e-04
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 553ms/step - exatidao: 0.7414 - loss: 0.5262 - precisao: 0.7291 - sensibilidade: 0.7598
Epoch 3: val_loss improved from 0.66101 to 0.36947, saving model to melhor_modelo_densenet.h5



Epoch 3: finished saving model to melhor_modelo_densenet.h5
500/500 ━━━━━━━━━━━━━━━━━━━━ 311s 623ms/step - exatidao: 0.7372 - loss: 0.5320 - precisao: 0.7289 - sensibilidade: 0.7552 - val_exatidao: 0.8582 - val_loss: 0.3695 - val_precisao: 0.9108 - val_sensibilidade: 0.9242 - learning_rate: 1.0000e-04
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 548ms/step - exatidao: 0.7530 - loss: 0.5087 - precisao: 0.7382 - sensibilidade: 0.7760
Epoch 4: val_loss did not improve from 0.36947
500/500 ━━━━━━━━━━━━━━━━━━━━ 306s 611ms/step - exatidao: 0.7543 - loss: 0.5115 - precisao: 0.7441 - sensibilidade: 0.7750 - val_exatidao: 0.7385 - val_loss: 0.5446 - val_precisao: 0.9297 - val_sensibilidade: 0.7501 - learning_rate: 1.0000e-04
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 544ms/step - exatidao: 0.7617 - loss: 0.4914 - precisao: 0.7512 - sensibilidade: 0.7856
Epoch 5: val_loss did not improve from 0.36947
500/500 ━━━━━━━━━━━━━━━━━━━━ 320s 608ms/step - exatidao: 0.7618 - loss: 0.4972 - precisao: 0.7522

In [5]:
from google.colab import files

print("A preparar o download do teu modelo de Inteligência Artificial...")
files.download('melhor_modelo_densenet.h5')

A preparar o download do teu modelo de Inteligência Artificial...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>